# LABORATÓRIO 10: O Pipeline Definitivo (RAG, QLoRA e Otimização de
Inferência na GPU)

**Objetivo:** Este é o laboratório integrador da disciplina. Vocês deverão orquestrar um
pipeline de IA ponta a ponta. O objetivo é simular um ambiente de produção
onde um modelo ajustado por QLoRA (Unidade II) precisa ler um contexto
gigantesco recuperado por um RAG (Unidade III), exigindo que vocês
manipulem a arquitetura do Self-Attention (Unidade I) utilizando FlashAttention
e KV Cache para evitar o erro de Out-Of-Memory (OOM) na GPU.

## Instalação de Dependências
Para este laboratório, além do `transformers` e `bitsandbytes`, precisaremos da biblioteca `flash-attn` para otimização de hardware.

In [ ]:
# Instalando dependencias usando uv

# Gerenciador de pacotes
!pip install -q uv

# Dependencias de IA
!uv pip install --system -q transformers accelerate bitsandbytes

# Compilando
!uv pip install --system -q flash-attn --no-build-isolation

## Passo 1: Ingestão Eficiente (QLORA) e Passo 2: Simulando o RAG gigantesco

Como não podemos carregar o LLM em 16-bits (Float16) pois ocuparia muita memoria, usaremos `bitsandbytes` para carregar o modelo em 4-bits.

Em seguida, vamos gerar um contexto imenso simulando os arquivos de `Isuldur (Senhor dos Aneis)`.

In [5]:
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Configurando QLORA (4-bits) seguindo documentacao: https://huggingface.co/docs/transformers/en/quantization/bitsandbytes?bnb=4-bit$0

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model_id  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Carregando o modelo base quantizado...")
model_unoptimised = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

# Passo 1 Metrica
memoria_alocada_mb = torch.cuda.memory_allocated() / (1024**2)
print(f"Memoria VRAM ocupada pelo modelo em 4-bits: {memoria_alocada_mb:.2f} MB")

# Criando contexto de O Senhor dos Aneis
lore_gondor = "Três Anéis para os Reis-Élficos sob o céu, Sete para os Senhores-Anões em seus salões de pedra, Nove para os Homens Mortais fadados a morrer, Um para o Senhor do Escuro em seu escuro trono Na Terra de Mordor onde as Sombras se deitam. "
# multiplicando a lore
texto_massivo_rag = lore_gondor * 20

inputs = tokenizer(texto_massivo_rag, return_tensors="pt").to("cuda")
print(f"Tamanho do contexto RAG recuperado: {inputs.input_ids.shape[1]} tokens.")

Carregando o modelo base quantizado...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Memoria VRAM ocupada pelo modelo em 4-bits: 1239.54 MB
Tamanho do contexto RAG recuperado: 1622 tokens.


## Passo 3: O Gargalo de Geração ( O Problema do Decoder )

Aqui vamos forçar o modelo a gerar 100 novos tokens lendo todo o contexto de O Senhor dos Anéis, mas **SEM usar cache de memória** (`use_cache = False`). Isso fará com que a complexidade $O(n^2)$ do Self-Attention calcule Q, K e V repetidamente para cada palavra gerada.

In [6]:
print("Iniciando geração NAO otimizada (O gargalo)")
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

# Gerando 100 tokens sem KV Cache
with torch.no_grad():
  outputs_unoptimised = model_unoptimised.generate(
      **inputs,
      max_new_tokens=100,
      use_cache=False,
      pad_token_id=tokenizer.eos_token_id,
  )

tempo_unoptimised        = time.time() - start_time
pico_memoria_unoptimised = torch.cuda.max_memory_allocated() / (1024**2)

print(f"Tempo de Geração (Sem Otimização): {tempo_unoptimised:.2f} segundos")
print(f"Pico de Memória VRAM (Sem Otimização): {pico_memoria_unoptimised:.2f} MB")

# Limpando a VRAM para o próximo passo
del model_unoptimised
del outputs_unoptimised
gc.collect()
torch.cuda.empty_cache()

Iniciando geração NAO otimizada (O gargalo)
Tempo de Geração (Sem Otimização): 87.46 segundos
Pico de Memória VRAM (Sem Otimização): 2159.91 MB


## Passo 4: A Engenharia de Otimização

Para este passo 4, recarregaremos o modelo ativando a otimização de hardware na memória SRAM da GPU (`flash_attention_2`) e habilitaremos o KV Cache na geração (`use_cache = True`) para salvar o nosso Transformer do colapso de memória.



In [7]:
print("Recarregando o modelo com FlashAttention-2")
model_optimised = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa", # Substituindo flash_attention_2 POR sdpa por ser
    torch_dtype=torch.float16
)

print("Iniciando geração OTIMIZADA...")
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

# Gerando 100 tokens COM KV Cache
with torch.no_grad():
    outputs_optimised = model_optimised.generate(
        **inputs,
        max_new_tokens=100,
        use_cache=True, # KV Cache ativado (Otimização de Software)
        pad_token_id=tokenizer.eos_token_id
    )

tempo_optimised = time.time() - start_time
pico_memoria_optimised = torch.cuda.max_memory_allocated() / (1024**2)

print(f"Tempo de Geração (Com Otimização): {tempo_optimised:.2f} segundos")
print(f"Pico de Memória VRAM (Com Otimização): {pico_memoria_optimised:.2f} MB")


`torch_dtype` is deprecated! Use `dtype` instead!


Recarregando o modelo com FlashAttention-2


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Iniciando geração OTIMIZADA...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Tempo de Geração (Com Otimização): 6.95 segundos
Pico de Memória VRAM (Com Otimização): 1602.64 MB
